In [1]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkAssignment") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [5]:
import os
print(os.listdir())

['.config', 'sample_data']


In [7]:
df = spark.read.csv(
    "data.csv",
    header=True,
    inferSchema=True
)

In [9]:
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [10]:
df.count()

9994

In [11]:
print("Total Rows Before:", df.count())

df = df.dropDuplicates()

print("Total Rows After:", df.count())

Total Rows Before: 9994
Total Rows After: 9994


In [12]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [13]:
df = df.na.drop()
print("Rows After Removing Null Values:", df.count())

Rows After Removing Null Values: 9994


In [14]:
tech_df = df.filter(df.Category == "Technology")

print("Technology Records:", tech_df.count())

tech_df.show(5)

Technology Records: 1847
+------+--------------+----------+----------+--------------+-----------+-----------------+---------+-------------+-------------+------------+-----------+------+---------------+----------+------------+--------------------+--------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|  Segment|      Country|         City|       State|Postal Code|Region|     Product ID|  Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+---------+-------------+-------------+------------+-----------+------+---------------+----------+------------+--------------------+--------+--------+--------+---------+
|  2001|CA-2017-166128| 4/11/2017| 4/18/2017|Standard Class|   LW-17215|       Luke Weiss| Consumer|United States|     Pasadena|  California|      91104|  West|TEC-AC-10001767|Technology| Accessorie

In [15]:
west_df = df.filter(df.Region == "West")

print("West Region Records:", west_df.count())
west_df.show(5)

West Region Records: 3203
+------+--------------+----------+----------+--------------+-----------+--------------+--------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+------+--------+--------+-------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID| Customer Name| Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name| Sales|Quantity|Discount| Profit|
+------+--------------+----------+----------+--------------+-----------+--------------+--------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+------+--------+--------+-------+
|  1142|CA-2014-146969| 9/29/2014| 10/3/2014|Standard Class|   AP-10915|Arthur Prichep|Consumer|United States|  Los Angeles|California|      90045|  West|FUR-FU-10004188|      Furniture| Furnishings|Luxo Professi

In [16]:
df.columns

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

In [17]:
df = df.withColumnRenamed("Order ID", "Order_ID")
df = df.withColumnRenamed("Customer ID", "Customer_ID")
df = df.withColumnRenamed("Product Name", "Product_Name")

In [18]:
df.columns

['Row ID',
 'Order_ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer_ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product_Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

In [19]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [20]:
from pyspark.sql.functions import col

df = df.withColumn(
    "Postal Code",
    col("Postal Code").cast("integer")
)

In [21]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [22]:
print("Total Rows:", df.count())

Total Rows: 9994


In [24]:
from pyspark.sql.functions import expr

df = df.withColumn(
    "Sales_Double",
    expr("try_cast(`Sales` as double)")
)

In [25]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .csv("data.csv")

In [26]:
df.select("Product Name", "Sales").show(20, False)

+----------------------------------------------------------------------------+--------+
|Product Name                                                                |Sales   |
+----------------------------------------------------------------------------+--------+
|Bush Somerset Collection Bookcase                                           |261.96  |
|Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back                 |731.94  |
|Self-Adhesive Address Labels for Typewriters by Universal                   |14.62   |
|Bretford CR4500 Series Slim Rectangular Table                               |957.5775|
|Eldon Fold 'N Roll Cart System                                              |22.368  |
|Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood            |48.86   |
|Newell 322                                                                  |7.28    |
|Mitel 5320 IP Phone VoIP phone                                              |907.152 |
|DXL Angle-View Binders with Loc

In [27]:
from pyspark.sql.functions import sum, avg, min, max

df.select(
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales"),
    min("Sales").alias("Minimum_Sales"),
    max("Sales").alias("Maximum_Sales")
).show()

print("Total Rows:", df.count())

+-----------------+-----------------+-------------+-------------+
|      Total_Sales|    Average_Sales|Minimum_Sales|Maximum_Sales|
+-----------------+-----------------+-------------+-------------+
|2297200.860299955|229.8580008304938|        0.444|       999.98|
+-----------------+-----------------+-------------+-------------+

Total Rows: 9994


In [28]:
from pyspark.sql.functions import count

df.groupBy("Category") \
  .agg(count("*").alias("Total_Records")) \
  .show()

+---------------+-------------+
|       Category|Total_Records|
+---------------+-------------+
|Office Supplies|         6026|
|      Furniture|         2121|
|     Technology|         1847|
+---------------+-------------+



In [29]:
from pyspark.sql.functions import sum, avg

df.groupBy("Category") \
  .agg(
      sum("Sales").alias("Total_Sales"),
      avg("Sales").alias("Average_Sales")
  ) \
  .show()

+---------------+-----------------+------------------+
|       Category|      Total_Sales|     Average_Sales|
+---------------+-----------------+------------------+
|Office Supplies|719047.0320000029|119.32410089611732|
|      Furniture|741999.7952999998|349.83488698727007|
|     Technology|836154.0329999966|452.70927612344155|
+---------------+-----------------+------------------+



In [30]:
df.groupBy("Region") \
  .agg(
      sum("Sales").alias("Total_Sales")
  ) \
  .show()

+-------+-----------------+
| Region|      Total_Sales|
+-------+-----------------+
|  South|391721.9050000003|
|Central|501239.8908000005|
|   East|678781.2399999979|
|   West|725457.8245000006|
+-------+-----------------+



In [34]:
result_df = df.groupBy("Category").count()
result_df.toPandas().to_csv("results.csv", index=False)